# 01 - Data feasibility 

**Question:** can we build leak-free snapshots and labels, and are there enough positive examples?

Parts
- A. deps.dev snapshot dates (which dates T can we use?)
- B. Parse the OSV npm advisories into a table
- C. Count positive labels per snapshot date T (the go / no-go numbers)
- D. Check the sample lockfiles (format and size)
- E. Check deps.dev table sizes BEFORE running any big query (free metadata calls)



In [ ]:
# ENVIRONMENT SETUP & REPOSITORY ROOT RESOLUTION

import sys
from pathlib import Path

# Traverse parent directories starting from the current working directory
# until the project root containing the 'ml' module directory is found
ROOT = Path.cwd().resolve()
while not (ROOT / 'ml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Add the repository root to sys.path so project modules can be imported
sys.path.insert(0, str(ROOT))

# Define canonical directory paths for datasets and pipeline artifacts
DATA = ROOT / 'data'
ARTIFACTS = ROOT / 'ml' / 'artifacts'

print('Project root:', ROOT)

Project root: D:\DepRadar


In [ ]:
# IMPORTS & GLOBAL CONFIGURATION

import json
import zipfile
import pandas as pd
import numpy as np

# Adjust Pandas display settings for wide tabular outputs in notebook cells
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

# ---- config ---
# Google Cloud project ID used for free BigQuery metadata inspections (Part E)
PROJECT_ID = 'depradar-data'      # replace with your Google Cloud PROJECT ID (shown in the console project picker)

# Toggle execution of BigQuery metadata and partition inspections
RUN_BIGQUERY = True               # set False to skip Part E

# Forward prediction window length: predict whether an advisory occurs within 6 months
HORIZON_MONTHS = 6                # label window length

# Candidate historical snapshot cutoff dates spaced at 6-month intervals
TARGET_DATES = ['2024-01-01', '2024-07-01', '2025-01-01', '2025-07-01', '2026-01-01']

# Paths to raw input datasets and processed interim storage
RAW_DEPS = DATA / 'raw' / 'deps_dev'
RAW_OSV = DATA / 'raw' / 'osv'
INTERIM = DATA / 'interim'

# Ensure the interim data output directory exists
INTERIM.mkdir(parents=True, exist_ok=True)

## Part A - deps.dev snapshot dates
`Snapshots` lists the timestamps at which deps.dev exported a full snapshot.
We pick the real snapshot closest to each target date, so T is a date that actually exists.

In [ ]:
# PART A: DEPS.DEV SNAPSHOT TIMESTAMPS & CUTOFF DATE MATCHING
# Purpose:
#   1. Load and inspect the available snapshot timestamps exported by deps.dev.
#   2. Verify snapshot frequency, continuity, and time bounds.
#   3. Map arbitrary calendar target dates (e.g., 2024-01-01) to the nearest
#      actual deps.dev snapshot timestamp (T) to prevent point-in-time leakage.
#   4. Construct the candidate evaluation schedule with corresponding 6-month
#      label observation windows [T, T + 6 months].

# Load the catalog of snapshot timestamps exported from deps.dev
snap = pd.read_parquet(RAW_DEPS / 'snapshots.parquet')
print(snap.dtypes)

# Automatically identify the datetime column representing snapshot timestamps
time_col = next(c for c in snap.columns if pd.api.types.is_datetime64_any_dtype(snap[c]))

# Ensure snapshot timestamps are timezone-aware UTC and chronologically ordered
snap[time_col] = pd.to_datetime(snap[time_col], utc=True)
snap = snap.sort_values(time_col).reset_index(drop=True)

# Inspect temporal coverage and compute interval gaps between consecutive snapshots
print(f'{len(snap)} snapshots, from {snap[time_col].min()} to {snap[time_col].max()}')
gaps = snap[time_col].diff().dt.days.dropna()
print('Gap between snapshots (days): min', gaps.min(), '| median', gaps.median(), '| max', gaps.max())

# Maps a target calendar date to the nearest snapshot timestamp actually in deps.dev
def nearest_snapshot(date):
    d = pd.Timestamp(date, tz='UTC')
    i = (snap[time_col] - d).abs().idxmin()
    return snap.loc[i, time_col]

# Resolve each candidate target date to the closest real snapshot timestamp T
T_LIST = [nearest_snapshot(d) for d in TARGET_DATES]

# Tabulate candidate cutoffs and compute their 6-month prediction window end dates
cand = pd.DataFrame({'target': TARGET_DATES, 'snapshot_T': T_LIST})
cand['window_end'] = cand['snapshot_T'] + pd.DateOffset(months=HORIZON_MONTHS)
cand

Time    datetime64[us, UTC]
dtype: object
230 snapshots, from 2022-05-08 02:05:29.029028+00:00 to 2026-10-05 21:01:16.215780+00:00
Gap between snapshots (days): min 1.0 | median 7.0 | max 17.0


,target,snapshot_T,window_end
0,2024-01-01,2024-01-01 21:01:38.824466+00:00,2024-07-01 21:01:38.824466+00:00
1,2024-07-01,2024-07-01 21:00:42.720159+00:00,2025-01-01 21:00:42.720159+00:00
2,2025-01-01,2024-12-30 21:00:45.311751+00:00,2025-06-30 21:00:45.311751+00:00
3,2025-07-01,2025-06-30 21:00:51.468070+00:00,2025-12-30 21:00:51.468070+00:00
4,2026-01-01,2025-12-29 21:00:46.883758+00:00,2026-06-29 21:00:46.883758+00:00


## Part B - OSV npm advisories -> one table
One row per (advisory, npm package). Saved to `data/interim/advisories_osv.parquet`.
We keep `ranges_json` and `versions_json` raw; the lockfile matcher will use them later.

In [ ]:
# PART B: OSV NPM ADVISORIES INGESTION & PARSING
# Purpose:
#   1. Extract and parse all JSON security advisories from raw/osv/npm_all.zip.
#   2. Flatten advisory records to one row per affected (advisory, npm package).
#   3. Standardize timestamps (published, modified, withdrawn) into UTC ISO8601.
#   4. Categorize advisories into 'malware' (MAL- prefix) vs 'vulnerability'
#      (GHSA, CVE, etc.) to allow clean separation in modeling.
#   5. Persist parsed tabular records to data/interim/advisories_osv.parquet.

# Parses a single OSV JSON advisory dictionary into flattened package-level rows
def parse_osv_record(rec):
    rows = []
    for aff in rec.get('affected', []):
        pkg = aff.get('package', {})
        # Only retain packages belonging to the npm ecosystem
        if pkg.get('ecosystem') != 'npm':
            continue
        rows.append({
            'osv_id': rec.get('id'),
            'package': pkg.get('name'),
            'published': rec.get('published'),
            'modified': rec.get('modified'),
            'withdrawn': rec.get('withdrawn'),
            'severity': (rec.get('database_specific') or {}).get('severity'),
            'aliases': json.dumps(rec.get('aliases', [])),
            'ranges_json': json.dumps(aff.get('ranges', [])),
            'versions_json': json.dumps(aff.get('versions', [])),
        })
    return rows

# Read the OSV npm zip archive and parse each advisory JSON file
rows, bad = [], 0
with zipfile.ZipFile(RAW_OSV / 'npm_all.zip') as z:
    names = [n for n in z.namelist() if n.endswith('.json')]
    print('JSON files in zip:', len(names))
    for n in names:
        try:
            rows.extend(parse_osv_record(json.loads(z.read(n))))
        except Exception as e:
            bad += 1

# Convert extracted records into a pandas DataFrame
adv = pd.DataFrame(rows)

# Parse ISO8601 timestamps with UTC timezone normalization
for c in ['published', 'modified', 'withdrawn']:
    adv[c] = pd.to_datetime(adv[c], utc=True, errors='coerce', format='ISO8601')

# Distinguish malicious package reports (MAL-*) from true vulnerability advisories
adv['kind'] = np.where(adv['osv_id'].str.startswith('MAL-'), 'malware', 'vulnerability')

print('unparseable files:', bad)
print('rows (advisory x package):', len(adv), '| distinct advisories:', adv.osv_id.nunique(), '| distinct packages:', adv.package.nunique())

# Cache the interim parsed DataFrame as a parquet file
adv.to_parquet(INTERIM / 'advisories_osv.parquet', index=False)
adv.head()

JSON files in zip: 230262
unparseable files: 0
rows (advisory x package): 233041 | distinct advisories: 230262 | distinct packages: 225731


,osv_id,package,published,modified,withdrawn,severity,aliases,ranges_json,versions_json,kind
0,EEF-CVE-2026-56812,phoenix,2026-07-07 15:22:46.933000+00:00,2026-09-24 21:30:06.586470584+00:00,NaT,NaN,"[""CVE-2026-56812"", ""GHSA-63mc-hw7g-86rr""]","[{""type"": ""SEMVER"", ""events"": [{""introduced"": ...",[],vulnerability
1,GHSA-2234-fmw7-43wr,hono,2024-10-15 17:43:50+00:00,2026-09-10 03:50:19.035733420+00:00,NaT,MODERATE,"[""CVE-2024-48913""]","[{""type"": ""SEMVER"", ""events"": [{""introduced"": ...",[],vulnerability
2,GHSA-2238-xc5r-v9hj,@tinacms/graphql,2026-03-12 17:50:28+00:00,2026-03-14 06:26:21.083736+00:00,NaT,MODERATE,"[""CVE-2026-24125""]","[{""type"": ""SEMVER"", ""events"": [{""introduced"": ...",[],vulnerability
3,GHSA-223g-f5mq-gw33,openlearnx,2026-05-13 01:39:04+00:00,2026-06-09 00:00:20.848304042+00:00,NaT,MODERATE,"[""CVE-2026-44720""]","[{""type"": ""SEMVER"", ""events"": [{""introduced"": ...",[],vulnerability
4,GHSA-223j-4rm8-mrmf,next,2025-04-02 22:35:37+00:00,2025-10-13 15:35:50+00:00,NaT,LOW,"[""CVE-2025-30218""]","[{""type"": ""SEMVER"", ""events"": [{""introduced"": ...","[""12.3.5""]",vulnerability


In [ ]:
# PART B (CONT.): OSV ADVISORY DATA PROFILING & DISTRIBUTION CHECKS
# Purpose:
#   1. Inspect advisory distribution by source ID prefix (GHSA, MAL, CVE, OSV).
#   2. Compare counts of malware reports vs standard vulnerability disclosures.
#   3. Check data quality: count withdrawn advisories and missing publish dates.
#   4. Compute annual counts of published, active (non-withdrawn) vulnerabilities.

# 1. Breakdown by advisory ID prefix (GHSA = GitHub Security, MAL = OpenSSF Malware, etc.)
print('--- by ID prefix')
print(adv.osv_id.str.extract(r'^([A-Za-z]+)')[0].value_counts())
print()

# 2. Breakdown by advisory kind: malware vs vulnerability
print('--- by kind (malware reports are a different thing from vulnerabilities)')
print(adv.kind.value_counts())
print()

# 3. Data hygiene checks: withdrawn status and missing publication timestamps
print('--- withdrawn:', adv.withdrawn.notna().sum(), '| missing published date:', adv.published.isna().sum())
print()

# 4. Annual volume of distinct published vulnerability advisories (excluding malware and withdrawn)
print('--- advisories published per year (vulnerabilities only)')
v = adv[(adv.kind == 'vulnerability') & adv.withdrawn.isna()]
print(v.drop_duplicates('osv_id').groupby(v.published.dt.year).size())

--- by ID prefix
0
MAL     222425
GHSA     10614
EEF          1
GSD          1
Name: count, dtype: int64

--- by kind (malware reports are a different thing from vulnerabilities)
kind
malware          222425
vulnerability     10616
Name: count, dtype: int64

--- withdrawn: 778 | missing published date: 0

--- advisories published per year (vulnerabilities only)
published
2017      46
2018     295
2019     358
2020     910
2021     578
2022     651
2023     387
2024     420
2025     606
2026    3187
dtype: int64


### Deep dive: 2026 vulnerability advisory surge and distribution
Anomalously high advisory counts appear in 2026. This analysis checks whether the surge is:
1. Concentrated in a small set of outlier packages or broadly distributed.
2. Composed of new entrant packages vs repeat offenders.
3. Accompanied by shifts in the severity rating distribution.

In [ ]:
# PART B DEEP DIVE: INVESTIGATING THE 2026 VULNERABILITY ADVISORY SURGE
# Purpose:
#   1. Trace the monthly volume of vulnerability advisories from 2024 onward to
#      observe the trajectory and timing of the surge.
#   2. Measure concentration in 2026: determine whether the surge is dominated by
#      a small number of heavily reported packages or distributed broadly.
#   3. Analyze package cohorts by year of FIRST vulnerability advisory to see if
#      the surge represents newly targeted packages or repeat offenders.
#   4. Evaluate severity composition across years to detect shifts in severity ratings.


# Filter for active (non-withdrawn) vulnerability records and extract month periods
v = adv[(adv.kind == 'vulnerability') & adv.withdrawn.isna()].copy()
v['month'] = v.published.dt.to_period('M')

# 1. Monthly count of distinct vulnerability advisories from January 2024 onward
print('--- distinct advisories per month, 2024-01 onward')
m = v.drop_duplicates('osv_id').groupby('month').size()
print(m[m.index >= pd.Period('2024-01', 'M')].to_string())

# 2. Package-level concentration analysis for 2026 advisories
print('\n--- 2026: how concentrated is it?')
v26 = v[v.published.dt.year == 2026]
per_pkg = v26.groupby('package').size().sort_values(ascending=False)
print('rows:', len(v26), '| distinct packages:', per_pkg.size,
      '| share of rows in the top 20 packages:', round(per_pkg.head(20).sum() / len(v26), 3))
print(per_pkg.head(15).to_string())

# 3. First-advisory cohort analysis: classify packages by the year they received their first advisory
print('\n--- packages by year of their FIRST vulnerability advisory')
first = v.groupby('package').published.min()
print(first.dt.year.value_counts().sort_index().to_string())

# 4. Severity distribution breakdown by publication year (normalized shares)
print('\n--- severity mix by year (share of advisories)')
sev = v.drop_duplicates('osv_id').assign(year=lambda d: d.published.dt.year)
print(pd.crosstab(sev.year, sev.severity.fillna('none'), normalize='index').round(2).to_string())

--- distinct advisories per month, 2024-01 onward
month
2024-01     29
2024-02     31
2024-03     41
2024-04     33
2024-05     39
2024-06     32
2024-07     38
2024-08     40
2024-09     43
2024-10     52
2024-11     17
2024-12     25
2025-01     25
2025-02     43
2025-03     51
2025-04     48
2025-05     29
2025-06     37
2025-07     53
2025-08     49
2025-09     95
2025-10     51
2025-11     55
2025-12     70
2026-01    136
2026-02    262
2026-03    603
2026-04    455
2026-05    361
2026-06    339
2026-07    325
2026-08    182
2026-09    291
2026-10    233
Freq: M

--- 2026: how concentrated is it?
rows: 4729 | distinct packages: 998 | share of rows in the top 20 packages: 0.436
package
openclaw                    590
n8n                         333
parse-server                155
electron                    155
next                         92
flowise                      81
axios                        76
undici                       69
vm2                          66
ghost        

C:\Users\HP\AppData\Local\Temp\ipykernel_34552\1451709509.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  v['month'] = v.published.dt.to_period('M')


## Part C - positive labels per snapshot date (the go / no-go numbers)

For each T: a package is **positive** if it gets a NEW advisory published in (T, T + 6 months].
Withdrawn advisories are excluded. Vulnerabilities and malware are counted separately.

Note: this counts positives only. The base rate needs the package universe from deps.dev (next step).

In [ ]:
# PART C: POSITIVE LABEL CALCULATION PER SNAPSHOT DATE T
# Purpose:
#   1. Compute label counts for each candidate snapshot cutoff date T over a
#      6-month forward horizon (T, T + 6 months].
#   2. Count how many packages receive at least one new advisory (positive label = 1).
#   3. Decompose positives into "first-time" positives (packages with no history of
#      advisories prior to T) vs "prior" positives (packages that already had advisories).
#   4. Separately track vulnerability and malware labels to evaluate model feasibility.

# Computes advisory statistics and positive label counts for a given cutoff date T
def label_stats(df, T, months):
    end = T + pd.DateOffset(months=months)
    out = {'T': T.date(), 'window_end': end.date()}
    for kind in ['vulnerability', 'malware']:
        # Filter for active records of the current kind with valid publication dates
        d = df[(df.kind == kind) & df.withdrawn.isna() & df.published.notna()]
        first_pub = d.groupby('package')['published'].min()
        
        # Advisories published strictly within the prediction window (T, T + months]
        in_window = d[(d.published > T) & (d.published <= end)]
        pos = set(in_window.package)
        
        # Packages with at least one historical advisory published on or before cutoff date T
        prior = set(d[d.published <= T].package)
        
        # Populate metrics for feasibility analysis
        out[f'{kind}: advisories in window'] = in_window.osv_id.nunique()
        out[f'{kind}: positive packages'] = len(pos)
        out[f'{kind}: positives with prior advisory'] = len(pos & prior)
        out[f'{kind}: positives that are first-time'] = len(pos - prior)
        out[f'{kind}: packages with any advisory <= T'] = len(prior)
    return out

# Calculate label feasibility statistics across all candidate snapshot dates in T_LIST
stats = pd.DataFrame([label_stats(adv, T, HORIZON_MONTHS) for T in T_LIST])
stats.T

,0,1,2,3,4
T,2024-01-01,2024-07-01,2024-12-30,2025-06-30,2025-12-29
window_end,2024-07-01,2025-01-01,2025-06-30,2025-12-30,2026-06-29
vulnerability: advisories in window,217,203,233,370,2161
vulnerability: positive packages,180,162,204,299,716
vulnerability: positives with prior advisory,52,54,72,108,220
vulnerability: positives that are first-time,128,108,132,191,496
vulnerability: packages with any advisory <= T,2375,2503,2611,2743,2931
malware: advisories in window,3942,4371,4969,186340,5437
malware: positive packages,3942,4371,4969,186340,5437
malware: positives with prior advisory,0,0,0,0,0


**How to read this**
- `positive packages` is the number of 1-labels you would have at that T. A few hundred or more is workable for LightGBM; a few dozen is not.
- `first-time` positives matter most: they are packages with no advisory history, so a model cannot just copy "had one before".
- If `malware` is large, keep it out of the labels (or model it separately).

## Part D - sample lockfiles

In [ ]:
# PART D: SAMPLE LOCKFILES FORMAT, SCHEMA & SIZE INSPECTION
# Purpose:
#   1. Verify that sample package-lock.json files are valid and parseable.
#   2. Inspect lockfileVersion (version 1 vs modern versions 2 and 3).
#   3. Confirm presence of top-level 'packages' key (standard in lockfile v2/v3).
#   4. Measure dependencies count, development dependency count, and file sizes.

lock_rows = []
# Iterate over all JSON lockfiles in the sample_lockfiles test directory
for p in sorted((DATA / 'sample_lockfiles').glob('*.json')):
    try:
        j = json.loads(p.read_text(encoding='utf-8'))
    except Exception as e:
        lock_rows.append({'file': p.name, 'error': str(e)[:60]})
        continue
        
    pk = j.get('packages')
    lock_rows.append({
        'file': p.name,
        'lockfileVersion': j.get('lockfileVersion'),
        'has_packages_key (v2/v3)': pk is not None,
        'n_packages': (len(pk) - 1) if pk else None,        # minus the root entry ''
        'n_dev_only': sum(1 for k, v in (pk or {}).items() if k and v.get('dev')),
        'size_kb': round(p.stat().st_size / 1024),
    })

pd.DataFrame(lock_rows)

,file,lockfileVersion,has_packages_key (v2/v3),n_packages,n_dev_only,size_kb
0,arcane-demo.json,3,True,205,152,102
1,digitalgov.json,2,True,999,272,805
2,expressjs-website.json,3,True,937,432,474
3,modernweb.json,3,True,3078,890,1450
4,wger.json,3,True,232,227,117


If a file shows `has_packages_key = False` it is an old v1 lockfile; the parser can skip it or use the older `dependencies` layout.

## Part E - deps.dev table sizes (free metadata calls, no query is run)
`client.get_table()` only reads table metadata. It does not scan data and costs nothing.
It shows each table's columns, rows and size so we can plan the export. Free quota: 1 TiB of queries per month.

In [ ]:
# PART E: BIGQUERY DEPS_DEV_V1 TABLE METADATA & SCHEMA INSPECTION
# Purpose:
#   1. Query Google Cloud BigQuery public metadata for all tables in deps_dev_v1.
#   2. Inspect row counts, total table sizes (in GB), and partitioning fields
#      without running data-scanning queries (free metadata API calls).
#   3. Print field names and types for each table to guide pipeline query design.

if RUN_BIGQUERY:
    from google.cloud import bigquery
    
    # Initialize BigQuery client using configured Google Cloud project
    client = bigquery.Client(project=PROJECT_ID)
    DATASET = 'bigquery-public-data.deps_dev_v1'
    TABLES = ['Snapshots', 'PackageVersions', 'Dependencies', 'DependencyGraphEdges',
              'Dependents', 'NPMRequirements', 'Advisories', 'Projects', 'PackageVersionToProject']
    
    summary, schemas = [], {}
    for t in TABLES:
        try:
            # client.get_table() makes a metadata-only request with 0 bytes billed
            tb = client.get_table(f'{DATASET}.{t}')
            summary.append({'table': t, 'type': tb.table_type, 'rows': tb.num_rows,
                            'size_GB': round((tb.num_bytes or 0) / 1e9, 2),
                            'partitioned_by': getattr(tb.time_partitioning, 'field', None)})
            schemas[t] = [(f.name, f.field_type) for f in tb.schema]
        except Exception as e:
            summary.append({'table': t, 'type': 'NOT FOUND / error', 'rows': None, 'size_GB': None,
                            'partitioned_by': str(e)[:60]})
            
    # Display table overview dataframe
    display(pd.DataFrame(summary))
    
    # Print schema columns and data types for each table
    for t, cols in schemas.items():
        print(f'\n{t}:', ', '.join(f'{n} ({ty})' for n, ty in cols))
else:
    print('Skipped (RUN_BIGQUERY = False)')

,table,type,rows,size_GB,partitioned_by
0,Snapshots,TABLE,231,0.00,NaN
1,PackageVersions,TABLE,22305642391,12917.06,SnapshotAt
2,Dependencies,TABLE,1263395694073,111836.21,SnapshotAt
3,DependencyGraphEdges,TABLE,2731358893436,344270.82,SnapshotAt
4,Dependents,TABLE,1085558261849,97468.59,SnapshotAt
5,NPMRequirements,TABLE,2990206020,1445.85,SnapshotAt
6,Advisories,TABLE,19224144,18.03,SnapshotAt
7,Projects,TABLE,866897779,100.17,SnapshotAt
8,PackageVersionToProject,TABLE,21813385316,2578.20,SnapshotAt



Snapshots: Time (TIMESTAMP)

PackageVersions: SnapshotAt (TIMESTAMP), System (STRING), Name (STRING), Version (STRING), Licenses (STRING), Links (RECORD), Advisories (RECORD), VersionInfo (RECORD), Hashes (RECORD), DependenciesProcessed (BOOLEAN), DependencyError (BOOLEAN), UpstreamPublishedAt (TIMESTAMP), Registries (STRING), SLSAProvenance (RECORD), UpstreamIdentifiers (RECORD), Purl (STRING), Attestations (RECORD), Description (STRING), Deprecated (STRING), ProjectStatus (STRING), ProjectStatusReason (STRING)

Dependencies: SnapshotAt (TIMESTAMP), System (STRING), Name (STRING), Version (STRING), Dependency (RECORD), MinimumDepth (INTEGER)

DependencyGraphEdges: SnapshotAt (TIMESTAMP), System (STRING), Name (STRING), Version (STRING), Requirement (STRING), From (RECORD), To (RECORD)

Dependents: SnapshotAt (TIMESTAMP), System (STRING), Name (STRING), Version (STRING), Dependent (RECORD), MinimumDepth (INTEGER), DependentIsHighestReleaseWithResolution (BOOLEAN)

NPMRequirements: Sna

### Snapshot partition availability across tables
Inspects available partition dates per table via `list_partitions` and tests data availability for each candidate cutoff date $T$ in `NPMRequirements`.

In [ ]:
# PART E (CONT.): TABLE PARTITIONS & CANDIDATE SNAPSHOT T DATA AVAILABILITY
# Purpose:
#   1. Query client.list_partitions() to find which partition dates exist in
#      each deps_dev_v1 table (free metadata call).
#   2. Perform dry-run queries against NPMRequirements for each candidate cutoff
#      date T to verify whether data is actually stored in that partition.
#      (Crucial finding: NPMRequirements only retains the ~40 most recent snapshots,
#      meaning earlier 2024 partitions return 0.00 GB and cannot be used for graph building).
#   3. Dry-run query scans for full dependency and unnested dependent aggregation queries.

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
DATASET = 'bigquery-public-data.deps_dev_v1'

# 1. Enumerate partitions in each table via metadata API
print('=== partitions that exist (free metadata call)')
for t in ['NPMRequirements', 'PackageVersions', 'Advisories', 'Projects', 'PackageVersionToProject']:
    try:
        parts = sorted(p for p in client.list_partitions(f'{DATASET}.{t}') if p[0].isdigit())
        print(f'{t:25s} {len(parts):4d} partitions | first {parts[0]} | last {parts[-1]}')
    except Exception as e:
        print(f'{t:25s} ERROR {str(e)[:150]}')

def ts(t):
    return t.strftime('%Y-%m-%d %H:%M:%S.%f+00')

def dry_gb(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=cfg).total_bytes_processed / 1e9

T_LATEST = snap[time_col].max()

# 2. Dry-run queries per candidate date T to verify data availability in NPMRequirements
print('\n=== NPMRequirements (Name, Version) per candidate T. 0.00 GB means that snapshot has no data')
for T in list(T_LIST) + [T_LATEST]:
    sql = f"SELECT Name, Version FROM `{DATASET}.NPMRequirements` WHERE SnapshotAt = TIMESTAMP('{ts(T)}')"
    print(f'{dry_gb(sql):8.2f} GB   {T.date()}')

# 3. Dry-run scan size estimation for more complex extraction queries at latest snapshot
print('\n=== bigger queries at the latest snapshot (dry run only)')
big = {
    'NPMRequirements @latest: Name, Version, Dependencies':
        f"SELECT Name, Version, Dependencies FROM `{DATASET}.NPMRequirements` WHERE SnapshotAt = TIMESTAMP('{ts(T_LATEST)}')",
    'Top packages by number of dependents (computed in BigQuery)':
        f"SELECT dep.Name, COUNT(DISTINCT r.Name) AS n_dependents FROM `{DATASET}.NPMRequirements` r, UNNEST(r.Dependencies) AS dep "
        f"WHERE r.SnapshotAt = TIMESTAMP('{ts(T_LATEST)}') GROUP BY dep.Name",
}
for name, sql in big.items():
    try:
        print(f'{dry_gb(sql):8.2f} GB   {name}')
    except Exception as e:
        print(f'   ERROR   {name}: {str(e)[:200]}')

=== partitions that exist (free metadata call)
NPMRequirements             41 partitions | first 20260105 | last 20261012
PackageVersions            233 partitions | first 20220508 | last 20261012
Advisories                 235 partitions | first 20220508 | last 20261012
Projects                   235 partitions | first 20220508 | last 20261012
PackageVersionToProject    234 partitions | first 20220508 | last 20261012

=== NPMRequirements (Name, Version) per candidate T. 0.00 GB means that snapshot has no data
    0.00 GB   2024-01-01
    0.00 GB   2024-07-01
    0.00 GB   2024-12-30
    0.00 GB   2025-06-30
    0.00 GB   2025-12-29
    3.75 GB   2026-10-05

=== bigger queries at the latest snapshot (dry run only)
   17.14 GB   NPMRequirements @latest: Name, Version, Dependencies
   11.16 GB   Top packages by number of dependents (computed in BigQuery)


### Schema nesting, clustering, and dry-run query scan costs
Inspects table clustering/partitioning requirements, detailed schema of nested record fields (`Dependencies`, `VersionInfo`, `Advisories`), and benchmarks projected scan sizes against the monthly free quota (1 TiB).

In [ ]:
# PART E (CONT.): BIGQUERY SCHEMA STRUCTURE & DRY-RUN QUERY COST ESTIMATION
# Purpose:
#   1. Inspect table partitioning keys, types, clustering fields, and partition
#      filter requirements for key deps_dev_v1 tables.
#   2. Inspect nested RECORD / REPEATED column structures (e.g., Dependencies,
#      DevDependencies, VersionInfo).
#   3. Benchmark exact byte scan volumes across candidate export queries using
#      BigQuery dry runs to verify they fit comfortably within the free 1 TiB monthly quota.

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
DATASET = 'bigquery-public-data.deps_dev_v1'

def show_fields(fields, indent=4):
    for f in fields:
        print(' ' * indent + f'{f.name} ({f.field_type}, {f.mode})')
        if f.field_type == 'RECORD':
            show_fields(f.fields, indent + 4)

# 1. Inspect table-level partitioning specifications and clustering columns
print('=== partitioning / clustering')
for t in ['NPMRequirements', 'PackageVersions', 'Advisories', 'Projects', 'PackageVersionToProject']:
    tb = client.get_table(f'{DATASET}.{t}')
    tp = tb.time_partitioning
    print(f'{t:25s} partition field: {getattr(tp, "field", None)} | type: {getattr(tp, "type_", None)}'
          f' | clustering: {tb.clustering_fields} | partition filter required: {getattr(tp, "require_partition_filter", None)}')

# 2. Inspect nested and repeated schema fields in depth
print('\n=== nested columns')
NESTED = {'NPMRequirements': ['Dependencies', 'DevDependencies'],
          'PackageVersions': ['VersionInfo', 'Advisories'],
          'Advisories': ['Packages']}
for t, cols in NESTED.items():
    tb = client.get_table(f'{DATASET}.{t}')
    for f in tb.schema:
        if f.name in cols:
            print(f'\n{t}.{f.name} ({f.mode})')
            show_fields(f.fields)

# 3. Dry runs: calculate projected query scan sizes against free monthly tier
print('\n=== dry runs: how many GB each export would scan (nothing is read or billed)')
def ts(t):
    return t.strftime('%Y-%m-%d %H:%M:%S.%f+00')

def dry_gb(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=cfg).total_bytes_processed / 1e9

T0 = T_LIST[2]                     # example snapshot (about 2024-12-30)
T_LATEST = snap[time_col].max()
tests = {
    'NPMRequirements @T0: Name, Version, Dependencies':
        f"SELECT Name, Version, Dependencies FROM `{DATASET}.NPMRequirements` WHERE SnapshotAt = TIMESTAMP('{ts(T0)}')",
    'NPMRequirements @T0: Name, Version only':
        f"SELECT Name, Version FROM `{DATASET}.NPMRequirements` WHERE SnapshotAt = TIMESTAMP('{ts(T0)}')",
    'PackageVersions @latest, NPM: Name, Version, UpstreamPublishedAt':
        f"SELECT Name, Version, UpstreamPublishedAt FROM `{DATASET}.PackageVersions` WHERE SnapshotAt = TIMESTAMP('{ts(T_LATEST)}') AND System = 'NPM'",
    'Advisories @latest (all columns)':
        f"SELECT * FROM `{DATASET}.Advisories` WHERE SnapshotAt = TIMESTAMP('{ts(T_LATEST)}')",
    'Projects @T0: Type, Name, StarsCount, ForksCount, OpenIssuesCount':
        f"SELECT Type, Name, StarsCount, ForksCount, OpenIssuesCount FROM `{DATASET}.Projects` WHERE SnapshotAt = TIMESTAMP('{ts(T0)}')",
    'PackageVersionToProject @T0, NPM: Name, Version, ProjectType, ProjectName':
        f"SELECT Name, Version, ProjectType, ProjectName FROM `{DATASET}.PackageVersionToProject` WHERE SnapshotAt = TIMESTAMP('{ts(T0)}') AND System = 'NPM'",
}
for name, sql in tests.items():
    try:
        print(f'{dry_gb(sql):10.2f} GB   {name}')
    except Exception as e:
        print(f'     ERROR   {name}: {str(e)[:200]}')
print('\nFree monthly quota is about 1,100 GB (1 TiB).')

=== partitioning / clustering
NPMRequirements           partition field: SnapshotAt | type: DAY | clustering: ['Name', 'Version'] | partition filter required: None
PackageVersions           partition field: SnapshotAt | type: DAY | clustering: ['System', 'Name'] | partition filter required: None
Advisories                partition field: SnapshotAt | type: DAY | clustering: ['Source', 'SourceID'] | partition filter required: None
Projects                  partition field: SnapshotAt | type: DAY | clustering: ['Type', 'Name'] | partition filter required: None
PackageVersionToProject   partition field: SnapshotAt | type: DAY | clustering: ['System', 'Name', 'Version'] | partition filter required: None

=== nested columns

NPMRequirements.Dependencies (REPEATED)
    Name (STRING, REQUIRED)
    Requirement (STRING, REQUIRED)

NPMRequirements.DevDependencies (REPEATED)
    Name (STRING, REQUIRED)
    Requirement (STRING, REQUIRED)

PackageVersions.Advisories (REPEATED)
    Source (STRING, R

### Results and Key Findings

1. Snapshot Timeline and Cadence
- deps.dev contains 230 weekly snapshots from May 2022 to October 2026, with an average gap of 7 days between snapshots.
- Five 6-month evaluation windows were selected between January 2024 and December 2025 without timeline gaps or future data leakage.

2. Security Advisories and Malware Split
- Successfully extracted 230,262 security advisories covering 225,731 distinct packages with zero unparseable files.
- 95.4% of records (222,425) are malicious spam packages (MAL prefix), while 4.6% (10,616) are genuine software vulnerabilities (GHSA prefix).
- Malware must be separated from software vulnerabilities during modeling to avoid skewed predictions.
- Vulnerability disclosures spiked in 2026 (3,187 reports), with 43.6% concentrated in the top 20 packages (such as openclaw, n8n, parse-server, electron, and next).
- Severity ratios remained steady across all years: around 11% Critical, 39% High, 42% Moderate, and 8% Low.

3. Label Feasibility for Machine Learning
- Each 6-month evaluation window contains between 162 and 716 positive packages with new vulnerabilities.
- Between 65% and 70% of positive packages in every window are first-time vulnerable packages with no prior security reports.
- There are sufficient positive cases to train and evaluate risk models without severe class scarcity.

4. Lockfile Compatibility
- All sample package-lock.json files use modern formats (version 2 and 3) with standard package dictionaries.
- Dependency trees can be parsed directly across projects ranging from 200 to over 3,000 packages.

5. BigQuery Data Limits and Costs
- Targeted export queries scan between 0.27 GB and 17.14 GB per run, remaining well below the 1,000 GB monthly free quota.
- Critical limit: NPMRequirements only stores data from January 2026 onwards (41 partitions). Queries for 2024 and 2025 return zero data.
- PackageVersions, Advisories, and Projects contain full historical partitions back to May 2022.
- For historical backtesting before 2026, dependency graphs must be parsed from lockfiles or version metadata rather than the NPMRequirements table.
